# 🔄 Challenge 15: 주문-결제-배송 Saga 패턴

**난이도:** ⭐⭐⭐⭐⭐  
**사전 요구사항:** 노트북 03, 08, 10  
**내비게이션:** [← 이전(14)](./14_challenge_bulk_processing.ipynb) | [다음(16) →](./16_challenge_realtime_delivery.ipynb)

---

## 📖 시나리오

### 문제 상황
11번가와 같은 이커머스 플랫폼에서 주문을 처리할 때, 다음과 같은 분산 트랜잭션 문제가 발생합니다:

1. **주문 생성** (Order Service)
2. **결제 처리** (Payment Service)
3. **재고 확인** (Inventory Service)
4. **배송 준비** (Shipping Service)

### Saga 패턴이란?
- 분산 트랜잭션을 여러 개의 로컬 트랜잭션으로 분해
- 각 단계가 실패하면 **보상 트랜잭션(Compensation)**을 실행
- 2PC(Two-Phase Commit)보다 확장성이 우수

### 실습 목표
- Saga Orchestrator 구현
- 4단계 주문 플로우 구축
- 실패 시 자동 보상 트랜잭션 실행

## 🏗️ 아키텍처

```
┌─────────────────────┐
│  SagaOrchestrator   │
└──────────┬──────────┘
           │
    ┌──────┴──────┬──────────┬──────────┐
    │             │          │          │
┌───▼───┐   ┌────▼────┐ ┌───▼────┐ ┌───▼─────┐
│ Step1 │   │  Step2  │ │ Step3  │ │  Step4  │
│ Order │   │ Payment │ │ Stock  │ │Shipping │
└───┬───┘   └────┬────┘ └───┬────┘ └────┬────┘
    │            │          │           │
    │  RabbitMQ  │  Redis   │           │
    └────────────┴──────────┴───────────┘
          │
    [Compensation]
    취소 ← 환불 ← 재고복구 ← 배송취소
```

## 🎯 적용 패턴

1. **Saga Orchestration** - 중앙 조정자가 플로우 관리
2. **Compensation Transaction** - 롤백 대신 보상 작업
3. **State Machine** - 각 단계의 상태 추적
4. **Idempotency** - 중복 실행 방지
5. **Event Sourcing** - 모든 상태 변경을 이벤트로 기록

In [ ]:
# 환경 설정
import httpx
import json
import asyncio
from datetime import datetime
from typing import Dict, List, Callable, Any
import time

# API 기본 URL
BASE_URL = "http://localhost:8000"

# Mock 데이터 로드
with open('../data/mock/saga_orders.json', 'r', encoding='utf-8') as f:
    MOCK_DATA = json.load(f)

print("✅ 환경 설정 완료")
print(f"📦 Mock 주문 데이터: {len(MOCK_DATA['orders'])}건")
print(f"📋 테스트 시나리오: {', '.join([o['order_id'] for o in MOCK_DATA['orders']])}")

## 🔧 Step 1: SagaOrchestrator 클래스 설계

### 핵심 기능
- `execute()`: 각 단계를 순차 실행
- `compensate()`: 실패 시 이전 단계들을 역순으로 보상
- Redis에 상태 추적
- RabbitMQ로 각 단계 이벤트 발행

In [ ]:
class SagaOrchestrator:
    """Saga 패턴 오케스트레이터"""
    
    def __init__(self, saga_id: str):
        self.saga_id = saga_id
        self.steps = []  # (step_name, execute_fn, compensate_fn)
        self.executed_steps = []  # 실행 완료된 단계 추적
        self.state = {}  # 공유 상태
    
    def add_step(self, name: str, execute_fn: Callable, compensate_fn: Callable):
        """Saga 단계 추가"""
        self.steps.append((name, execute_fn, compensate_fn))
    
    async def execute(self) -> Dict[str, Any]:
        """Saga 실행 - 모든 단계를 순차 실행"""
        print(f"\n🚀 Saga 시작: {self.saga_id}")
        print("=" * 60)
        
        for idx, (name, execute_fn, compensate_fn) in enumerate(self.steps, 1):
            try:
                print(f"\n[{idx}/{len(self.steps)}] 단계 실행: {name}")
                
                # 단계 실행
                result = await execute_fn(self.state)
                self.executed_steps.append((name, compensate_fn))
                
                # Redis에 상태 저장
                await self._save_state(name, "COMPLETED", result)
                
                print(f"   ✅ {name} 완료")
                
            except Exception as e:
                print(f"   ❌ {name} 실패: {e}")
                
                # 실패 시 보상 트랜잭션 실행
                await self.compensate()
                
                return {
                    "status": "FAILED",
                    "failed_at": name,
                    "error": str(e),
                    "compensated": True
                }
        
        print("\n" + "=" * 60)
        print("✅ Saga 완료!")
        return {"status": "SUCCESS", "state": self.state}
    
    async def compensate(self):
        """보상 트랜잭션 - 실행된 단계들을 역순으로 롤백"""
        print("\n🔄 보상 트랜잭션 시작...")
        print("=" * 60)
        
        # 역순으로 보상
        for name, compensate_fn in reversed(self.executed_steps):
            try:
                print(f"   ⏪ {name} 보상 중...")
                await compensate_fn(self.state)
                await self._save_state(name, "COMPENSATED", {})
                print(f"   ✅ {name} 보상 완료")
            except Exception as e:
                print(f"   ⚠️ {name} 보상 실패: {e}")
        
        print("=" * 60)
        print("✅ 보상 트랜잭션 완료")
    
    async def _save_state(self, step: str, status: str, data: Dict):
        """Redis에 Saga 상태 저장"""
        async with httpx.AsyncClient() as client:
            key = f"saga:{self.saga_id}:{step}"
            value = json.dumps({
                "status": status,
                "timestamp": datetime.now().isoformat(),
                "data": data
            })
            await client.post(
                f"{BASE_URL}/redis/kv/set",
                json={"key": key, "value": value}
            )

print("✅ SagaOrchestrator 클래스 정의 완료")

## 📝 Step 2: 4단계 정의

### 각 단계별 작업
1. **주문 생성** - Redis에 주문 저장
2. **결제 처리** - 결제 정보 저장 + RabbitMQ 발행
3. **재고 확인** - 재고 체크 + 재고 차감
4. **배송 준비** - 배송 정보 발행

### 각 단계별 보상 작업
1. 주문 취소
2. 결제 환불
3. 재고 복구
4. 배송 취소

In [ ]:
# Step 1: 주문 생성
async def create_order(state: Dict) -> Dict:
    """주문 생성 단계"""
    async with httpx.AsyncClient() as client:
        order_data = state['order_data']
        
        # Redis에 주문 저장
        key = f"order:{order_data['order_id']}"
        await client.post(
            f"{BASE_URL}/redis/kv/set",
            json={"key": key, "value": json.dumps(order_data)}
        )
        
        state['order_id'] = order_data['order_id']
        return {"order_id": order_data['order_id']}

async def cancel_order(state: Dict):
    """주문 취소 보상"""
    async with httpx.AsyncClient() as client:
        key = f"order:{state['order_id']}"
        await client.delete(f"{BASE_URL}/redis/kv/delete/{key}")


# Step 2: 결제 처리
async def process_payment(state: Dict) -> Dict:
    """결제 처리 단계"""
    async with httpx.AsyncClient() as client:
        payment_data = {
            "order_id": state['order_id'],
            "amount": state['order_data']['total_amount'],
            "method": state['order_data']['payment_method'],
            "timestamp": datetime.now().isoformat()
        }
        
        # Redis에 결제 정보 저장
        key = f"payment:{state['order_id']}"
        await client.post(
            f"{BASE_URL}/redis/kv/set",
            json={"key": key, "value": json.dumps(payment_data)}
        )
        
        # RabbitMQ에 결제 완료 이벤트 발행
        await client.post(
            f"{BASE_URL}/rabbitmq/direct/publish",
            params={"queue_name": "payment-events"},
            json={
                "content": "payment.completed",
                "metadata": payment_data
            }
        )
        
        state['payment_id'] = f"PAY-{state['order_id']}"
        return payment_data

async def refund_payment(state: Dict):
    """결제 환불 보상"""
    async with httpx.AsyncClient() as client:
        # 환불 기록
        refund_data = {
            "order_id": state['order_id'],
            "refunded_at": datetime.now().isoformat()
        }
        key = f"refund:{state['order_id']}"
        await client.post(
            f"{BASE_URL}/redis/kv/set",
            json={"key": key, "value": json.dumps(refund_data)}
        )


# Step 3: 재고 확인
async def reserve_inventory(state: Dict) -> Dict:
    """재고 확인 및 예약 단계"""
    async with httpx.AsyncClient() as client:
        items = state['order_data']['items']
        
        # 재고 체크 시뮬레이션
        for item in items:
            # Redis에서 재고 확인
            stock_key = f"stock:{item['product_id']}"
            response = await client.get(
                f"{BASE_URL}/redis/kv/get/{stock_key}"
            )
            
            data = response.json()
            if not data.get('exists'):
                # 재고 정보 없음 - 초기값 설정
                current_stock = 100
            else:
                current_stock = int(data['value'])
            
            # 재고 부족 시 실패
            if current_stock < item['quantity']:
                raise Exception(f"재고 부족: {item['product_name']} (필요: {item['quantity']}, 현재: {current_stock})")
            
            # 재고 차감
            new_stock = current_stock - item['quantity']
            await client.post(
                f"{BASE_URL}/redis/kv/set",
                json={"key": stock_key, "value": str(new_stock)}
            )
        
        state['reserved_items'] = items
        return {"reserved": len(items)}

async def release_inventory(state: Dict):
    """재고 복구 보상"""
    async with httpx.AsyncClient() as client:
        for item in state.get('reserved_items', []):
            stock_key = f"stock:{item['product_id']}"
            response = await client.get(
                f"{BASE_URL}/redis/kv/get/{stock_key}"
            )
            
            data = response.json()
            if data.get('exists'):
                current_stock = int(data['value'])
                new_stock = current_stock + item['quantity']
                await client.post(
                    f"{BASE_URL}/redis/kv/set",
                    json={"key": stock_key, "value": str(new_stock)}
                )


# Step 4: 배송 준비
async def arrange_shipping(state: Dict) -> Dict:
    """배송 준비 단계"""
    async with httpx.AsyncClient() as client:
        shipping_data = {
            "order_id": state['order_id'],
            "address": state['order_data']['shipping_address'],
            "status": "READY",
            "created_at": datetime.now().isoformat()
        }
        
        # RabbitMQ에 배송 준비 이벤트 발행
        await client.post(
            f"{BASE_URL}/rabbitmq/direct/publish",
            params={"queue_name": "shipping-events"},
            json={
                "content": "shipping.ready",
                "metadata": shipping_data
            }
        )
        
        state['shipping_id'] = f"SHIP-{state['order_id']}"
        return shipping_data

async def cancel_shipping(state: Dict):
    """배송 취소 보상"""
    async with httpx.AsyncClient() as client:
        cancel_data = {
            "order_id": state['order_id'],
            "status": "CANCELLED"
        }
        await client.post(
            f"{BASE_URL}/rabbitmq/direct/publish",
            params={"queue_name": "shipping-events"},
            json={
                "content": "shipping.cancelled",
                "metadata": cancel_data
            }
        )

print("✅ 4단계 정의 완료")
print("   - create_order / cancel_order")
print("   - process_payment / refund_payment")
print("   - reserve_inventory / release_inventory")
print("   - arrange_shipping / cancel_shipping")

## ✅ Step 3: 정상 시나리오 실행

SAGA-001: 모든 단계가 성공하는 경우

In [ ]:
# SAGA-001: 성공 시나리오
success_order = MOCK_DATA['orders'][0]

print(f"📦 주문 정보: {success_order['order_id']}")
print(f"   고객: {success_order['customer_name']}")
print(f"   상품: {', '.join([item['product_name'] for item in success_order['items']])}")
print(f"   금액: {success_order['total_amount']:,}원")
print()

# Saga 생성
saga = SagaOrchestrator(success_order['order_id'])
saga.state['order_data'] = success_order

# 단계 추가
saga.add_step("주문생성", create_order, cancel_order)
saga.add_step("결제처리", process_payment, refund_payment)
saga.add_step("재고확인", reserve_inventory, release_inventory)
saga.add_step("배송준비", arrange_shipping, cancel_shipping)

# 실행
result = await saga.execute()

print("\n📊 최종 결과:")
print(json.dumps(result, indent=2, ensure_ascii=False))

## ❌ Step 4: 실패 시나리오 + 보상 트랜잭션

SAGA-002: 재고 부족으로 실패 → 자동 보상

In [ ]:
# SAGA-002: 실패 시나리오 (재고 부족)
fail_order = MOCK_DATA['orders'][1]

print(f"📦 주문 정보: {fail_order['order_id']}")
print(f"   고객: {fail_order['customer_name']}")
print(f"   상품: {', '.join([item['product_name'] for item in fail_order['items']])}")
print(f"   ⚠️ 예상: 재고 부족으로 실패 예정")
print()

# 재고를 의도적으로 부족하게 설정
async with httpx.AsyncClient() as client:
    for item in fail_order['items']:
        stock_key = f"stock:{item['product_id']}"
        await client.post(
            f"{BASE_URL}/redis/kv/set",
            json={"key": stock_key, "value": "1"}  # 재고 1개만
        )

# Saga 생성
saga2 = SagaOrchestrator(fail_order['order_id'])
saga2.state['order_data'] = fail_order

# 단계 추가
saga2.add_step("주문생성", create_order, cancel_order)
saga2.add_step("결제처리", process_payment, refund_payment)
saga2.add_step("재고확인", reserve_inventory, release_inventory)
saga2.add_step("배송준비", arrange_shipping, cancel_shipping)

# 실행
result = await saga2.execute()

print("\n📊 최종 결과:")
print(json.dumps(result, indent=2, ensure_ascii=False))

## 🔍 검증

Redis에서 두 시나리오의 최종 상태 확인

In [ ]:
async def verify_saga_state(order_id: str):
    """Saga 상태 검증"""
    async with httpx.AsyncClient() as client:
        print(f"\n🔍 {order_id} 상태 확인")
        print("=" * 60)
        
        steps = ["주문생성", "결제처리", "재고확인", "배송준비"]
        
        for step in steps:
            key = f"saga:{order_id}:{step}"
            response = await client.get(
                f"{BASE_URL}/redis/kv/get/{key}"
            )
            
            result = response.json()
            if result.get('exists') and result.get('value'):
                data = json.loads(result['value'])
                status_icon = "✅" if data['status'] == "COMPLETED" else "🔄"
                print(f"{status_icon} {step}: {data['status']} ({data['timestamp'][:19]})")
            else:
                print(f"❌ {step}: 데이터 없음")

# 두 시나리오 검증
await verify_saga_state(success_order['order_id'])
await verify_saga_state(fail_order['order_id'])

## 💡 핵심 포인트

### 1. Saga vs 2PC (Two-Phase Commit)
| 항목 | Saga | 2PC |
|------|------|-----|
| 확장성 | 높음 | 낮음 |
| 일관성 | Eventual | Strong |
| 복잡도 | 높음 (보상 로직) | 중간 |
| 장애 허용 | 우수 | 낮음 |

### 2. Idempotency (멱등성)
- 같은 요청을 여러 번 실행해도 결과가 동일
- 보상 트랜잭션도 멱등성 보장 필요
- Redis에 실행 기록 저장으로 중복 방지

### 3. State Machine
```
PENDING → ORDER_CREATED → PAYMENT_COMPLETED → INVENTORY_RESERVED → SHIPPING_READY
   ↓           ↓                  ↓                     ↓                  ↓
FAILED    ORDER_CANCELLED   PAYMENT_REFUNDED    INVENTORY_RELEASED  SHIPPING_CANCELLED
```

### 4. 프로덕션 고려사항
- **타임아웃 설정**: 각 단계별 최대 실행 시간
- **재시도 전략**: Exponential Backoff
- **Dead Letter Queue**: 최종 실패 처리
- **모니터링**: 각 단계별 성공률, 평균 소요 시간

## 🚀 확장 아이디어

### 1. Saga Log
```python
# 모든 단계를 이벤트로 기록
event_store = [
    {"step": "주문생성", "status": "STARTED", "timestamp": "..."},
    {"step": "주문생성", "status": "COMPLETED", "timestamp": "..."},
    ...
]
```

### 2. Dead Letter 처리
- 보상 트랜잭션도 실패 시 → Manual intervention queue로 이동
- 관리자가 수동으로 처리

### 3. Saga 모니터링 대시보드
- 실시간 Saga 실행 현황
- 각 단계별 성공률, 평균 소요 시간
- 보상 트랜잭션 발생 빈도

### 4. Saga Choreography
- Orchestration 대신 각 서비스가 이벤트를 구독하고 다음 단계 트리거
- 더 낮은 결합도, 하지만 플로우 추적이 어려움

In [ ]:
# 정리: Redis 데이터 삭제
async def cleanup():
    async with httpx.AsyncClient() as client:
        # Saga 상태 삭제
        for order in MOCK_DATA['orders']:
            order_id = order['order_id']
            steps = ["주문생성", "결제처리", "재고확인", "배송준비"]
            
            for step in steps:
                key = f"saga:{order_id}:{step}"
                await client.delete(f"{BASE_URL}/redis/kv/delete/{key}")
            
            # 주문/결제/환불 데이터 삭제
            for prefix in ["order", "payment", "refund"]:
                await client.delete(
                    f"{BASE_URL}/redis/kv/delete/{prefix}:{order_id}"
                )

await cleanup()
print("🧹 정리 완료")

## 🧭 내비게이션

[← 이전 (14)](#) | [다음 (16) →](./16_challenge_realtime_delivery.ipynb)

---

**학습 완료!** 🎉  
다음 노트북에서는 실시간 배달 알림 시스템을 구축합니다.